In [8]:
import sys
sys.path.append("..")

In [9]:
import tqdm
import os
import time
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import cvxpy as cp
from copy import deepcopy

from src.data import *
from src.model import *
from src.recourse import *
from src.utils import *

warnings.filterwarnings('ignore')

In [10]:
def append_result(d, algorithm, seed, alpha, lamb, i, x_0, theta_0, x_r, theta_r=None):
    d["algorithm"].append(algorithm)
    d["seed"].append(seed)
    d["alpha"].append(alpha)
    d["lambda"].append(lamb)
    d["i"].append(i)
    d["x_0"].append(x_0.round(4))
    d["x_r"].append(x_r.round(4))
    d["theta_0"].append(theta_0.round(4))

def append_result_time(d_time, index, e_time):
    d_time["i"].append(index)
    d_time["e_time"].append(np.float64(e_time).round(5))

In [11]:
def recourse_runner(seed: int, X: np.ndarray, recourse: Recourse, params: dict, dataset: Dataset):
    alpha = params['alpha']
    lamb = params['lamb']
    recourse_i = params['recourse_index']
    f_name = f'../results/recourse/lr_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl'
    f_name_time = f'../results/recourse_time/lr_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}_time.pkl'
    
    results = {'algorithm': [], 'seed': [], 'alpha': [], 'lambda': [], 'i': [], 'x_0': [], 'x_r': [], 'theta_0': []}
    results_time = {'i': [], "e_time": []}

    weights_0, bias_0 = recourse.weights, recourse.bias
    theta_0 = np.hstack((weights_0, bias_0))
    if recourse.name == "ROAR":
        print(weights_0, bias_0, theta_0)
    n = len(X)

    for i in tqdm.trange(n, desc=f'[{recourse.name}] [alpha={alpha}] [lambda={lamb}]', colour='#0091ff'):
        x_0 = X[i]
        
        start_time = time.perf_counter()
        x_r = recourse.get_recourse(x_0)
        end_time = time.perf_counter()
        elapsed_time = end_time - start_time
        
        append_result(results, recourse.name, seed, alpha, lamb, recourse_i[i], x_0, theta_0, x_r)
        append_result_time(results_time, recourse_i[i], elapsed_time)

    df_results = pd.DataFrame(results)
    df_results_time = pd.DataFrame(results_time)

    if params['append_results'] and os.path.exists(f_name):
        df_tmp = pd.read_pickle(f_name)
        df_results = pd.concat((df_tmp, df_results), axis=0).sort_values(['i'], ignore_index=True)
    if params['append_results'] and params['save_time'] and os.path.exists(f_name_time):
        df_time_tmp = pd.read_pickle(f_name_time)
        df_results_time = pd.concat((df_time_tmp, df_results_time), axis=0).sort_values(['i'], ignore_index=True)        

    if params["save_results"]:
        print(f'[{recourse.name}] Saving results for {dataset.name} run {seed}')
        df_results.to_pickle(f_name)
    if params["save_time"]:
        print(f'[{recourse.name}] Saving time results for {dataset.name} run {seed}')
        df_results_time.to_pickle(f_name_time)
    
    return df_results

In [ ]:
def run_experiment(dataset: Dataset, recourse_fns: List[Recourse], params: dict, results: List):
    alpha = params['alpha']
    lamb = params['lamb']
    
    for seed in params['seeds']:
        train_data, test_data = dataset.get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data
        
        base_model = LR()
        base_model.train(X_train.values, y_train.values)
        
        weights_0 = base_model.model.coef_[0]
        bias_0 = base_model.model.intercept_
        
        recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)
        
        for recourse_fn in recourse_fns:
            recourse_needed_X_test_idx = np.arange(recourse_needed_X_test.shape[0])

            recourse = recourse_fn(weights=weights_0, bias=bias_0, alpha=alpha, lamb=lamb)
            if params["lamb"] is None:
                params['lamb'] = recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
                recourse.lamb = params['lamb']
            
            # if params['append_results']:
            #     f_name = f"../results/recourse/lr_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl"
            #     if os.path.exists(f_name):
            #         df_tmp = pd.read_pickle(f_name)
            #         recourse_needed_X_test_idx = np.setdiff1d(recourse_needed_X_test_idx, df_tmp['i'].to_numpy())
            #     else:
            #         print(f"{f_name} does not exist. The recourses will be created in a new file")

            #     if recourse_needed_X_test_idx.size == 0:
            #         print(f"{f_name} already has all the recourses. Skipping")
            #         continue
            
            # if params['subsample']:
            #     rng = np.random.default_rng(seed=seed)
            #     size_N = int(np.rint(params['subsample_size'] * recourse_needed_X_test.shape[0]))
            #     if recourse_needed_X_test_idx.shape[0] < size_N:
            #         size_N = recourse_needed_X_test_idx.shape[0]
            #     recourse_needed_X_test_idx = rng.choice(recourse_needed_X_test_idx, size=size_N, replace=False)

            if params['sba_fix']:
                prev_indexes = {0: np.array([0,1,2,19,20,24,25,31,32]),
                                1: np.array([4,14,15,16,17,25,27,29,34]),
                                2: np.array([3,4,5,10,11,18,30,31,32]),
                                3: np.array([2,3,6,7,21,25,26,27,30]),
                                4: np.array([2,22,26,30,31,33,34,35,36])}
                
                f_name = f"../results/recourse/lr_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl"
                if os.path.exists(f_name):
                    df_tmp = pd.read_pickle(f_name)
                    recourse_needed_X_test_idx = np.setdiff1d(prev_indexes[seed], df_tmp['i'].to_numpy())
                else:
                    print("SOMETHING WENT WRONG")

                rng = np.random.default_rng(seed=seed)
                if recourse_needed_X_test_idx.size < 3:
                    recourse_needed_X_test_idx = rng.choice(recourse_needed_X_test_idx, size=recourse_needed_X_test_idx.size, replace=False)
                else:
                    recourse_needed_X_test_idx = rng.choice(recourse_needed_X_test_idx, size=3, replace=False)

            params['recourse_index'] = recourse_needed_X_test_idx.copy()
            recourse_needed_X_test = recourse_needed_X_test[recourse_needed_X_test_idx]
                
            df_results = recourse_runner(seed, recourse_needed_X_test, recourse, params, dataset)
            results.append(df_results)

lr_sba(alph=0.1) <br>
Seed0 - 0,1,2,19,20,24,25,31,32 <br>
Seed1 - 4,14,15,16,17,25,27,29,34 <br>
Seed2 - 3,4,5,10,11,18,30,31,32 <br>
Seed3 - 2,3,6,7,21,25,26,27,30 <br>
Seed4 - 2,22,26,30,31,33,34,35,36 <br>
<br>
lr_sba(alpha=0.5) <br>
Seed0 - 32,24 <br>
Seed1 - 16,18 <br>
Seed2 - 10,32 <br>
Seed3 - 3,28 <br>
Seed4 - 26,35 <br>

In [42]:
np.hstack((np.arange(0.001, 0.0105, 0.001), np.arange(0.02, 0.105, 0.01), np.arange(0.2, 0.55, 0.1))).round(5)

array([0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009,
       0.01 , 0.02 , 0.03 , 0.04 , 0.05 , 0.06 , 0.07 , 0.08 , 0.09 ,
       0.1  , 0.2  , 0.3  , 0.4  , 0.5  ])

In [ ]:
alphas = [0.5] # <------------------------
lambdas = [0.01, 0.1, 1.4, 2.1, 3.5] # <------------------------

torch.manual_seed(0)

for lamb in lambdas:
    for alpha in alphas:

        d_results = {}
        params = {}
        params['alpha'] = alpha # float, None
        params['lamb'] = lamb
        params['seeds'] = range(5)
        params['save_results'] = True
        params['append_results'] = True
        params['subsample'] = False
        params['subsample_size'] = 0.5
        params['save_time'] = True

        params['sba_fix'] = True

        datasets = [SBADataset()] # <------------------------
        recourse_fns = [L1Recourse] # <------------------------

        for dataset in datasets:
            results = []
            print(f'Running {dataset.name} data...')
            run_experiment(dataset, recourse_fns, params, results)
            
            d_results[dataset.name] = pd.concat(results)
            print(f'Finished {dataset.name}\n')

Running sba data...


[L1PSD] [alpha=0.5] [lambda=0.01]:   0%|          | 0/3 [00:04<?, ?it/s]


KeyboardInterrupt: 